# 02 — Preprocessing and validation scheme
**Owner: Emmanuel Gyamfi Kwarteng (Preprocessing & Validation)**

* Cleaning and encoding live in `src/preprocess.py` so every notebook uses the same steps.
* Anything learned from data (scaler means, one-hot categories) is inside an sklearn
  `Pipeline`, so it is fitted **on the training fold only** — no leakage into validation.

In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from src.preprocess import (load_data, clean, feature_columns, linear_preprocessor,
                            tree_frame, get_folds, TARGET)

train, test = load_data()
train_c, test_c = clean(train), clean(test)
train_c[feature_columns()].head()

,Age,Flight Distance,Departure Delay in Minutes,Arrival Delay in Minutes,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Gender,Customer Type,Type of Travel,Class,arrival_delay_missing,n_not_applicable
0,36,694,0,0.0,4,4,4,3,1,4,...,4,4,3,1,0,1,0,Eco,0,0
1,56,651,0,0.0,5,5,5,5,5,4,...,5,5,4,3,1,1,1,Business,0,0
2,31,1371,0,0.0,3,4,2,5,3,3,...,3,4,5,3,0,1,0,Eco,0,0
3,10,1616,0,0.0,3,5,3,3,3,3,...,5,3,4,3,1,1,0,Eco,0,0
4,23,481,5,0.0,3,4,3,4,5,3,...,4,5,3,5,0,0,1,Eco,0,0


## 1. Metric
The sample submission contains a **probability** for each passenger
(constant 0.4436 = the share of satisfied passengers), so submissions are scored with
**ROC AUC** — the area under the ROC curve.

*In plain words:* pick one satisfied and one unsatisfied passenger at random. AUC is the
probability that the model gives the satisfied one a higher score. 0.5 = random guessing,
1.0 = perfect ranking. It rewards **ranking passengers correctly**, not hitting a 0.5 threshold,
and it is not affected by the 44/56 class imbalance.

> *Check the Evaluation tab on Kaggle to confirm the metric before the report.*

## 2. Encoding and scaling
| Feature group | Linear models | Tree models |
|---|---|---|
| Gender, Customer Type, Type of Travel | fixed 0/1 mapping | same |
| Class (3 levels) | one-hot | ordered integer Eco=0 < Eco Plus=1 < Business=2 |
| Delays | `log1p` then standardised | raw |
| Age, distance, ratings, `n_not_applicable` | standardised | raw (trees don't need scaling) |

In [2]:
pre = linear_preprocessor()
Xt = pre.fit_transform(train_c[feature_columns()])
print("Linear-model matrix:", Xt.shape)
print(pd.DataFrame(Xt[:3], columns=pre.get_feature_names_out()).round(2).T.head(25))

Linear-model matrix: (699635, 25)
                                           0     1     2
delays__Departure Delay in Minutes     -0.29 -0.29 -0.29
delays__Arrival Delay in Minutes       -0.28 -0.28 -0.28
num__Age                               -0.22  1.23 -0.58
num__Flight Distance                   -0.69 -0.74  0.02
num__Inflight wifi service              0.95  1.73  0.17
num__Departure/Arrival time convenient  0.57  1.25  0.57
num__Ease of Online booking             0.89  1.64 -0.61
num__Gate location                     -0.01  1.58  1.58
num__Food and drink                    -1.74  1.33 -0.20
num__Online boarding                    0.49  0.49 -0.28
num__Seat comfort                      -2.01  0.34 -0.45
num__Inflight entertainment            -1.88  1.18 -0.35
num__On-board service                  -0.42  1.20  0.39
num__Leg room service                   0.41  1.20 -0.37
num__Baggage handling                   0.18  1.11  0.18
num__Checkin service                   -0.35  0.48  1.

In [3]:
X_tree = tree_frame(train_c)
print("Tree-model matrix:", X_tree.shape)
X_tree.head()

Tree-model matrix: (699635, 23)


,Age,Flight Distance,Departure Delay in Minutes,Arrival Delay in Minutes,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Gender,Customer Type,Type of Travel,Class,arrival_delay_missing,n_not_applicable
0,36,694,0,0.0,4,4,4,3,1,4,...,4,4,3,1,0,1,0,0,0,0
1,56,651,0,0.0,5,5,5,5,5,4,...,5,5,4,3,1,1,1,2,0,0
2,31,1371,0,0.0,3,4,2,5,3,3,...,3,4,5,3,0,1,0,0,0,0
3,10,1616,0,0.0,3,5,3,3,3,3,...,5,3,4,3,1,1,0,0,0,0
4,23,481,5,0.0,3,4,3,4,5,3,...,4,5,3,5,0,0,1,0,0,0


## 3. Validation scheme
**5-fold stratified cross-validation** with a fixed seed.
* 700k rows is a lot, so 5 folds give stable scores (each fold ≈ 140k rows).
* Stratification keeps the 44/56 class balance in every fold.
* Rows are independent passengers with no time order, so random folds are appropriate.
* The same `FOLDS` are reused for every model, so the comparison table is fair.
* Out-of-fold (OOF) AUC is reported as the main validation score; test predictions are the
  average of the 5 fold models.

In [4]:
FOLDS = get_folds(train_c[TARGET].values)   # defined in src/preprocess.py
for i, (tr_idx, va_idx) in enumerate(FOLDS):
    print(f"fold {i}: train {len(tr_idx):,}  valid {len(va_idx):,}  "
          f"valid satisfied share {train_c[TARGET].iloc[va_idx].mean():.4f}")

fold 0: train 559,708  valid 139,927  valid satisfied share 0.4436
fold 1: train 559,708  valid 139,927  valid satisfied share 0.4436
fold 2: train 559,708  valid 139,927  valid satisfied share 0.4436
fold 3: train 559,708  valid 139,927  valid satisfied share 0.4436
fold 4: train 559,708  valid 139,927  valid satisfied share 0.4436


Sanity check: AUC of a constant prediction must be exactly 0.5 on every fold.

In [5]:
y = train_c[TARGET].values
print([roc_auc_score(y[va], np.full(len(va), y[tr].mean())) for tr, va in FOLDS])

[0.5, 0.5, 0.5, 0.5, 0.5]
